# Quality Control

**Remove low-quality cells and uninformative genes**

---

## Purpose

Apply stringent quality control to remove:
- Low-quality nuclei (low gene detection)
- Potential doublets (abnormally high gene counts)
- Dying cells (high mitochondrial expression)
- Uninformative genes (expressed in <3 cells)

---

## Workflow

1. Load raw h5ad file
2. Calculate QC metrics
3. Visualize distributions (before filtering)
4. Apply filtering thresholds
5. Visualize distributions (after filtering)
6. Save filtered h5ad

---

## Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════
DATASET = 'psychad_aging'  # CHANGE THIS FOR EACH COHORT

# Options:
# - 'psychad_aging'
# - 'psychad_ad'
# - 'psychencode'
# - 'mathys'
# - 'australian'

RANDOM_SEED = 42
FIGURE_DPI = 300

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# QUALITY CONTROL THRESHOLDS
# ═══════════════════════════════════════════════════════════════════════════════

# Cell filtering
MIN_GENES = 200          # Minimum genes detected per cell
MAX_GENES = 8000         # Maximum genes (remove potential doublets)
MAX_MT_PERCENT = 20      # Maximum % mitochondrial reads

# Gene filtering
MIN_CELLS = 3            # Minimum cells expressing gene

print("QC Thresholds:")
print(f"  Cells: {MIN_GENES} ≤ n_genes ≤ {MAX_GENES}")
print(f"  Cells: pct_counts_mt ≤ {MAX_MT_PERCENT}%")
print(f"  Genes: expressed in ≥ {MIN_CELLS} cells")

---

## Imports

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(RANDOM_SEED)

# Scanpy settings
sc.settings.verbosity = 1
sc.settings.set_figure_params(
    dpi=FIGURE_DPI,
    dpi_save=FIGURE_DPI,
    frameon=False,
    figsize=(6, 6)
)

print(f"Scanpy version: {sc.__version__}")
print(f"Random seed: {RANDOM_SEED}")

---

## Paths

In [ ]:
# Base directories
BASE_DIR = Path('..').resolve().parent
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
FIGURES_DIR = BASE_DIR / 'figures' / '00_preprocessing' / DATASET

# Create output directories
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Input/Output files
INPUT_FILE = RAW_DIR / f'{DATASET}_raw.h5ad'
OUTPUT_FILE = PROCESSED_DIR / f'01_qc_filtered_{DATASET}.h5ad'

print("Paths:")
print(f"  Input:  {INPUT_FILE}")
print(f"  Output: {OUTPUT_FILE}")
print(f"  Figures: {FIGURES_DIR}")

---

## Load Data

In [ ]:
print("="*70)
print("LOADING RAW DATA")
print("="*70)

adata = sc.read_h5ad(INPUT_FILE)

print(f"\nDataset: {DATASET}")
print(f"Cells: {adata.n_obs:,}")
print(f"Genes: {adata.n_vars:,}")
print(f"\nMetadata columns: {list(adata.obs.columns)}")
print(f"\nMatrix type: {type(adata.X)}")
print(f"Data type: {adata.X.dtype}")

---

## Calculate QC Metrics

In [ ]:
print("="*70)
print("CALCULATING QC METRICS")
print("="*70)

# Identify mitochondrial genes (starts with 'MT-')
adata.var['mt'] = adata.var_names.str.startswith('MT-')
n_mt_genes = adata.var['mt'].sum()
print(f"\nMitochondrial genes: {n_mt_genes}")

# Calculate QC metrics
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=['mt'],
    percent_top=None,
    log1p=False,
    inplace=True
)

print("\nQC metrics calculated:")
print("  - n_genes_by_counts: Number of genes detected")
print("  - total_counts: Total UMI counts")
print("  - pct_counts_mt: % mitochondrial expression")

# Summary statistics
print("\nBefore filtering:")
print(f"  n_genes_by_counts: {adata.obs['n_genes_by_counts'].median():.0f} (median)")
print(f"  total_counts: {adata.obs['total_counts'].median():.0f} (median)")
print(f"  pct_counts_mt: {adata.obs['pct_counts_mt'].median():.2f}% (median)")

---

## Visualize QC Metrics (Before Filtering)

In [ ]:
print("="*70)
print("VISUALIZING QC METRICS (BEFORE FILTERING)")
print("="*70)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle(f'{DATASET} - QC Metrics Before Filtering', fontsize=14, fontweight='bold')

# Row 1: Histograms
axes[0, 0].hist(adata.obs['n_genes_by_counts'], bins=100, color='#4A90E2', alpha=0.7, edgecolor='black')
axes[0, 0].axvline(MIN_GENES, color='red', linestyle='--', linewidth=2, label=f'Min: {MIN_GENES}')
axes[0, 0].axvline(MAX_GENES, color='red', linestyle='--', linewidth=2, label=f'Max: {MAX_GENES}')
axes[0, 0].set_xlabel('n_genes_by_counts')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()

axes[0, 1].hist(np.log10(adata.obs['total_counts']), bins=100, color='#50C878', alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('log10(total_counts)')
axes[0, 1].set_ylabel('Frequency')

axes[0, 2].hist(adata.obs['pct_counts_mt'], bins=100, color='#E24A4A', alpha=0.7, edgecolor='black')
axes[0, 2].axvline(MAX_MT_PERCENT, color='red', linestyle='--', linewidth=2, label=f'Max: {MAX_MT_PERCENT}%')
axes[0, 2].set_xlabel('pct_counts_mt')
axes[0, 2].set_ylabel('Frequency')
axes[0, 2].legend()

# Row 2: Scatter plots
axes[1, 0].scatter(
    adata.obs['total_counts'],
    adata.obs['n_genes_by_counts'],
    s=1, alpha=0.3, c='#4A90E2'
)
axes[1, 0].set_xlabel('total_counts')
axes[1, 0].set_ylabel('n_genes_by_counts')
axes[1, 0].set_xscale('log')

axes[1, 1].scatter(
    adata.obs['total_counts'],
    adata.obs['pct_counts_mt'],
    s=1, alpha=0.3, c='#50C878'
)
axes[1, 1].set_xlabel('total_counts')
axes[1, 1].set_ylabel('pct_counts_mt')
axes[1, 1].set_xscale('log')
axes[1, 1].axhline(MAX_MT_PERCENT, color='red', linestyle='--', linewidth=2)

axes[1, 2].scatter(
    adata.obs['n_genes_by_counts'],
    adata.obs['pct_counts_mt'],
    s=1, alpha=0.3, c='#E24A4A'
)
axes[1, 2].set_xlabel('n_genes_by_counts')
axes[1, 2].set_ylabel('pct_counts_mt')
axes[1, 2].axvline(MIN_GENES, color='red', linestyle='--', linewidth=1)
axes[1, 2].axvline(MAX_GENES, color='red', linestyle='--', linewidth=1)
axes[1, 2].axhline(MAX_MT_PERCENT, color='red', linestyle='--', linewidth=2)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_qc_before_filtering.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / '01_qc_before_filtering.svg', bbox_inches='tight')
plt.show()

print("✓ QC plots saved")

---

## Apply Filtering Thresholds

In [ ]:
print("="*70)
print("APPLYING FILTERS")
print("="*70)

# Store original counts
n_cells_before = adata.n_obs
n_genes_before = adata.n_vars

print(f"\nBefore filtering:")
print(f"  Cells: {n_cells_before:,}")
print(f"  Genes: {n_genes_before:,}")

# Filter cells
print(f"\nFiltering cells...")
sc.pp.filter_cells(adata, min_genes=MIN_GENES)
print(f"  After min_genes filter: {adata.n_obs:,} cells")

# Max genes filter
adata = adata[adata.obs['n_genes_by_counts'] <= MAX_GENES, :].copy()
print(f"  After max_genes filter: {adata.n_obs:,} cells")

# Mitochondrial filter
adata = adata[adata.obs['pct_counts_mt'] <= MAX_MT_PERCENT, :].copy()
print(f"  After pct_mt filter: {adata.n_obs:,} cells")

# Filter genes
print(f"\nFiltering genes...")
sc.pp.filter_genes(adata, min_cells=MIN_CELLS)
print(f"  After min_cells filter: {adata.n_vars:,} genes")

# Summary
n_cells_after = adata.n_obs
n_genes_after = adata.n_vars

cells_removed = n_cells_before - n_cells_after
genes_removed = n_genes_before - n_genes_after

print(f"\nAfter filtering:")
print(f"  Cells: {n_cells_after:,} ({cells_removed:,} removed, {cells_removed/n_cells_before*100:.1f}%)")
print(f"  Genes: {n_genes_after:,} ({genes_removed:,} removed, {genes_removed/n_genes_before*100:.1f}%)")

---

## Visualize QC Metrics (After Filtering)

In [ ]:
print("="*70)
print("VISUALIZING QC METRICS (AFTER FILTERING)")
print("="*70)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle(f'{DATASET} - QC Metrics After Filtering', fontsize=14, fontweight='bold')

# Row 1: Histograms
axes[0, 0].hist(adata.obs['n_genes_by_counts'], bins=100, color='#4A90E2', alpha=0.7, edgecolor='black')
axes[0, 0].set_xlabel('n_genes_by_counts')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title(f'n={len(adata.obs):,} cells')

axes[0, 1].hist(np.log10(adata.obs['total_counts']), bins=100, color='#50C878', alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('log10(total_counts)')
axes[0, 1].set_ylabel('Frequency')

axes[0, 2].hist(adata.obs['pct_counts_mt'], bins=100, color='#E24A4A', alpha=0.7, edgecolor='black')
axes[0, 2].set_xlabel('pct_counts_mt')
axes[0, 2].set_ylabel('Frequency')

# Row 2: Scatter plots
axes[1, 0].scatter(
    adata.obs['total_counts'],
    adata.obs['n_genes_by_counts'],
    s=1, alpha=0.3, c='#4A90E2'
)
axes[1, 0].set_xlabel('total_counts')
axes[1, 0].set_ylabel('n_genes_by_counts')
axes[1, 0].set_xscale('log')

axes[1, 1].scatter(
    adata.obs['total_counts'],
    adata.obs['pct_counts_mt'],
    s=1, alpha=0.3, c='#50C878'
)
axes[1, 1].set_xlabel('total_counts')
axes[1, 1].set_ylabel('pct_counts_mt')
axes[1, 1].set_xscale('log')

axes[1, 2].scatter(
    adata.obs['n_genes_by_counts'],
    adata.obs['pct_counts_mt'],
    s=1, alpha=0.3, c='#E24A4A'
)
axes[1, 2].set_xlabel('n_genes_by_counts')
axes[1, 2].set_ylabel('pct_counts_mt')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_qc_after_filtering.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / '01_qc_after_filtering.svg', bbox_inches='tight')
plt.show()

print("✓ QC plots saved")

---

## Summary Statistics

In [ ]:
print("="*70)
print("SUMMARY STATISTICS")
print("="*70)

summary_stats = {
    'Metric': ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
    'Mean': [
        adata.obs['n_genes_by_counts'].mean(),
        adata.obs['total_counts'].mean(),
        adata.obs['pct_counts_mt'].mean()
    ],
    'Median': [
        adata.obs['n_genes_by_counts'].median(),
        adata.obs['total_counts'].median(),
        adata.obs['pct_counts_mt'].median()
    ],
    'Std': [
        adata.obs['n_genes_by_counts'].std(),
        adata.obs['total_counts'].std(),
        adata.obs['pct_counts_mt'].std()
    ]
}

summary_df = pd.DataFrame(summary_stats)
print("\n" + summary_df.to_string(index=False))

# Save summary
summary_df.to_csv(FIGURES_DIR / '01_qc_summary_stats.csv', index=False)
print("\n✓ Summary statistics saved")

---

## Save Filtered Data

In [ ]:
print("="*70)
print("SAVING FILTERED DATA")
print("="*70)

# Save
adata.write(OUTPUT_FILE)

print(f"\n✓ Saved: {OUTPUT_FILE}")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")
print(f"  Size: {OUTPUT_FILE.stat().st_size / 1e9:.2f} GB")

print("\n" + "="*70)
print("✓ QUALITY CONTROL COMPLETE")
print("="*70)